# 34_alarm_chatbot_JIW

- 목적: 경보 근거 카드(32)와 결과 요약(28·33)을 **대화로 묻고 답하는 챗봇 인터페이스**를 만들고, 오픈소스 소형 LLM을 이 PC에서 직접 돌려 쓰는 방식이 안전하고 쓸 만한지 점검한다. 설계: `docs/chatbot_design_JIW.md`.
- 입력: `results/28_model_ensemble_JIW`, `results/32_alarm_explain_JIW/cards.csv`, `results/33_early_warning_JIW`의 CSV. 탐지 모델 가중치는 쓰지 않는다.
- LLM: 허깅페이스 `Qwen/Qwen2.5-1.5B-Instruct`(Apache-2.0, 파라미터 1.54B)를 `models/34_alarm_chatbot_JIW/`에 내려받아 **로컬 CPU**에서 실행한다. 외부로 아무것도 보내지 않는다.
- 출력: `results/34_alarm_chatbot_JIW/`(facts.csv · eval_offline.csv · hf_generate_examples.csv · routing_eval.csv · eval_hf_hybrid.csv · guardrail_test.csv)
- 담당: JIW
- 탐지 대상: 유압펌프 모터-펌프 구동부 이상 (센서 부착 부위 미확정이라 원인을 단정하지 않는다)
- 심사 대응: 4번(경보 → 조치 설명 인터페이스), 5번(근거 기반 LLM 적용)
- 근거 표기: **[데이터]** 이 노트북 출력 / **[내 판독]** 출력을 내가 읽고 판단한 것 / **[가정]** 운영 전제

## 먼저 한계

- 챗봇은 **판정하지 않는다.** 경보 여부는 28번 규칙(코드)이 정하고, 챗봇은 그 근거를 풀어 설명한다. 챗봇 오류가 탐지 성능(AUC·오경보율)에 영향을 주지 않는다.
- 소형 모델(1.5B)의 한국어 성능은 제한적이다. 이 노트북은 **모델이 문장을 새로 쓰게 하면 틀린다**는 것(§3)을 확인하고, 모델의 역할을 **근거 번호 고르기**로 줄인 설계(§4)를 평가한다.
- 질문 세트는 내가 만들었고, 근거 선택 프롬프트를 이 세트로 두 번 고쳤다(v1 → v2 + 도메인 단어 게이트). **적중률은 낙관적이다.**
- 모델 파일(2.9GB)은 git에 없다. 없으면 HF 구간은 저장된 CSV를 읽기만 한다.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
NB = "34_alarm_chatbot_JIW"  # 파일명(확장자 제외)과 같게
FIG, RES = paths.nb_dirs(NB)
SEED = 42
np.random.seed(SEED)
import chatbot_jiw as C
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 160)
HF_OK = (C.MODEL_DIR / "model.safetensors").exists()      # 없으면 C.download_model()로 한 번 내려받는다(약 3GB)
RERUN_HF = HF_OK                                           # 모델이 있으면 다시 돌리고, 없으면 results의 CSV를 읽는다
print(NB, RES.relative_to(ROOT), "| 로컬 모델:", "있음" if HF_OK else "없음(저장된 결과 CSV만 사용)")

34_alarm_chatbot_JIW results\34_alarm_chatbot_JIW | 로컬 모델: 있음


## 1. 구조

```
질문 ─▶ 카드 질문? ──예──▶ 근거 카드 문장 + 해석 제한 문구
        │아니오
        ▼
       원인 단정·시점 예측 질문? ──예──▶ 규칙이 거절하고 이유(F11·F15)를 댄다   ← 정책은 모델에 맡기지 않는다
        │아니오
        ▼
       키워드 규칙에 걸림? ──예──▶ 해당 근거 문장
        │아니오 (hybrid 모드)
        ▼
       도메인 단어 게이트 ──없음──▶ "제공된 근거에 없습니다."
        │있음
        ▼
       로컬 LLM이 근거 번호만 고름 ─▶ 그 근거 문장 그대로 출력

FACTS = 결과 CSV에서 읽은 수치 문장 17개(F1~F17). 모든 답은 [F#]로 인용한다.
```

- 모델은 **문장을 쓰지 않고 번호만 고른다.** 답은 항상 근거 문장 그대로라 새 오류를 만들 수 없다. 틀릴 수 있는 것은 "엉뚱한 근거를 고르거나 못 고르는 것"뿐이고, 그건 눈에 보인다.
- 안전장치(코드로 검사): 답의 숫자가 FACTS·CARD에 없으면 경고, 원인·시점 단정 표현을 정규식으로 찾아 경고, 모델 답이 검사를 어기면 근거 문장으로 대체한다.

In [2]:
facts = C.build_facts()
cards = C.load_cards()
ft = pd.Series(facts, name="내용").rename_axis("번호").to_frame()
ft.to_csv(RES / "facts.csv")
print(len(facts), "개 facts, 카드", len(cards), "건 (실제 이상 창", int((cards.label == 1).sum()), "/ 정상 오경보 창", int((cards.label == 0).sum()), ")")
ft

17 개 facts, 카드 64 건 (실제 이상 창 63 / 정상 오경보 창 1 )


,내용
번호,
F1,"경보 규칙: 주의 = CNN 점수가 임계를 넘음, 경보 = CNN과 MCD 점수가 모두 임계를 넘음. 경보는 항상 주의의 부분집합이다. CNN은 신호의 다음 값을 예측해 틀린 정도를 보고, MCD는 윈도우 통계 18개가 정상 무리에서 멀리 떨어진 정도를 본다."
F2,임계는 학습용 정상 점수의 상위 1%(q99)로만 정하고 이상 데이터로 조정하지 않는다. 점수는 임계로 나눈 배수로 표시한다(1배가 임계).
F3,"CNN 단독(주의 단계) 세그먼트 오경보율은 2.8% / 1.8%(group_kfold / time_block), 시간당 약 10.0 / 6.3건이다. 실제 이상 세그먼트는 전부 탐지했다."
F4,"CNN과 MCD를 모두 넘어야 하는 경보 단계의 세그먼트 오경보율은 0.2% / 0.1%, 시간당 약 0.8 / 0.3건이다. 실제 이상 세그먼트는 전부 탐지했다(미탐 0건)."
F5,"MCD 단독 세그먼트 오경보율은 3.8% / 3.1%, CNN 또는 MCD(OR)는 6.4% / 4.8%로 오경보가 늘어 채택하지 않았다."
F6,"약한 이상 탐지: 합성 강도 2에서 경보(AND)의 탐지율은 CNN 단독 대비 32% / 44%(group_kfold / time_block)만 유지되고, 강도 8에서는 87% / 94%다. 합성 강도 1 이하의 약한 이상은 경보도 주의도 거의 잡지 못한다(CNN 단독 강도 1 ..."
F7,"오경보는 고부하 운전 구간에 몰린다. CNN 단독 고부하 오경보 3.2% / 2.2%, 저부하 0.1% / 0.0%(group_kfold / time_block). 운전 상태별 임계로 바꿔도 고부하 오경보는 3.1% / 3.0%로 줄지 않아 채택하지 않았다. 경보 단계(CNN A..."
F8,"조기경보(합성 열화 시나리오): 열화가 완성되기 전에 알린 선행 시간 중앙값은 주의 1.39 / 3.60 / 7.39분, 경보 0.98 / 2.81 / 6.71분이다(열화 2 / 5 / 10분 기준). 선행 시간은 열화 기간의 약 70%라서 실제 설비의 열화 속도를 모르면 '몇 ..."
F9,"정상 구간에서 거짓 사건은 주의 시간당 6.2건, 경보 시간당 0.3건이다(정상 약 3.1시간 분량, 사건 수가 적어 불확실하다)."


## 2. 오프라인 백엔드 점검 (키워드 규칙 + 템플릿)

질문 14개: 답해야 하는 10개(필수 인용), 단정을 거절해야 하는 2개(시점 예측, 원인 단정), 범위 밖 2개.

In [3]:
ev = C.evaluate(C.OfflineBackend(), facts, cards)
ev.drop(columns=["모델 원답"]).to_csv(RES / "eval_offline.csv", index=False)
print("통과", int(ev["통과"].sum()), "/", len(ev))
ev[["질문", "종류", "인용", "필수 인용 충족", "거절", "통과"]]

통과 14 / 14


,질문,종류,인용,필수 인용 충족,거절,통과
0,경보랑 주의는 뭐가 달라?,answer,"F1,F2,F3,F4",True,False,True
1,임계값은 어떻게 정해?,answer,"F1,F2,F3,F4",True,False,True
2,오경보는 얼마나 자주 나?,answer,"F3,F4,F7,F9",True,False,True
3,고부하에서 오경보가 왜 많아?,answer,"F3,F4,F7,F9",True,False,True
4,약한 이상도 잡아?,answer,"F6,F16,F17",True,False,True
5,조기경보는 몇 분 전에 울려?,answer,"F8,F11,F16",True,False,True
6,미탐은 어떤 경우야?,answer,"F6,F16,F17",True,False,True
7,어떻게 점검해야 해?,answer,"F14,F15",True,False,True
8,이 결과를 얼마나 믿을 수 있어?,answer,"F12,F15,F16",True,False,True
9,CNN 단독이랑 MCD 단독 비교해줘,answer,"F3,F4,F5",True,False,True


**발견 [데이터]** 14개 모두 통과한다. 키워드 규칙이 내가 쓴 질문 표현을 알고 있어서 나온 결과라 대화 성능의 증거가 아니다. 보장하는 것은 답이 근거 문장 그대로라 숫자가 어긋나지 않는다는 것, 원인·시점 질문에 거절 근거가 붙는다는 것, 범위 밖 질문에 "근거에 없음"으로 답한다는 것이다.

## 3. 실험: 모델이 문장을 새로 쓰게 하면 (generate 모드)

로컬 LLM에게 질문과 관련된 근거만 주고 답을 **자유롭게 쓰게** 한다. 같은 14개 질문 중 모델로 가는 9개의 원답과, 코드 검사(숫자·단정 표현) 결과, 내가 읽은 정오를 비교한다.

In [4]:
GEN_Q = ["경보랑 주의는 뭐가 달라?", "임계값은 어떻게 정해?", "오경보는 얼마나 자주 나?", "고부하에서 오경보가 왜 많아?", "약한 이상도 잡아?",
         "미탐은 어떤 경우야?", "어떻게 점검해야 해?", "이 결과를 얼마나 믿을 수 있어?", "CNN 단독이랑 MCD 단독 비교해줘"]
HF_PATH = RES / "hf_generate_examples.csv"
if RERUN_HF:
    gen = C.HFBackend(mode="generate", max_new_tokens=250)
    allowed = C.facts_text(facts)
    rows = []
    for q in GEN_Q:
        a = gen.answer(q, [], facts, None)
        rows.append({"질문": q, "경로": gen.last_route, "모델 원답": a, "숫자 검사 위반": len(C.verify_numbers(a, allowed)), "단정 표현": len(C.unsafe_claims(a))})
    gdf = pd.DataFrame(rows)
else:
    gdf = pd.read_csv(HF_PATH)
# 내 판독: 출력(탐욕 디코딩이라 같은 입력이면 같은 답)을 읽고 근거 문장과 대조해 적은 것
READ = {
    "경보랑 주의는 뭐가 달라?": "오류 — 주의를 'CNN과 MCD 중 하나라도 넘으면'으로 설명(실제 주의는 CNN 단독)",
    "임계값은 어떻게 정해?": "정확",
    "오경보는 얼마나 자주 나?": "부분 — 수치는 맞으나 경보 단계 수치를 빼고 '자주 나타납니다'로 표현",
    "고부하에서 오경보가 왜 많아?": "오류 — 수치는 맞으나 원인 서술이 뒤섞임",
    "약한 이상도 잡아?": "오류 — 첫 문장이 '잡을 수 있습니다'로 근거(거의 못 잡음)와 반대",
    "미탐은 어떤 경우야?": "오류 — '합성 열화 또는 합성 이상일 때 발생'이라고 틀리게 일반화",
    "어떻게 점검해야 해?": "부분 — 권고 목록은 맞으나 '이상 가능성을 최소화' 같은 근거에 없는 문장 추가",
    "이 결과를 얼마나 믿을 수 있어?": "오류 — 한계 근거(F16)가 있는데 '근거에 없습니다'로 과도하게 거절",
    "CNN 단독이랑 MCD 단독 비교해줘": "오류 — CNN 단독 항목에 경보(AND) 수치 0.2% / 0.1%를 붙임",
}
gdf["내 판독"] = gdf["질문"].map(READ)
gdf.to_csv(HF_PATH, index=False)
for _, r in gdf.iterrows():
    print("Q. " + r["질문"] + "   [숫자 위반 " + str(r["숫자 검사 위반"]) + ", 단정 " + str(r["단정 표현"]) + "]  내 판독: " + r["내 판독"])
    print("A. " + r["모델 원답"] + "\n")
kinds = gdf["내 판독"].str.split(" — ").str[0].value_counts()
print("내 판독 집계:", kinds.to_dict(), "| 코드 검사가 잡은 것:", int(((gdf["숫자 검사 위반"] > 0) | (gdf["단정 표현"] > 0)).sum()), "/", len(gdf))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Q. 경보랑 주의는 뭐가 달라?   [숫자 위반 0, 단정 0]  내 판독: 오류 — 주의를 'CNN과 MCD 중 하나라도 넘으면'으로 설명(실제 주의는 CNN 단독)
A. 경보와 주의는 모두 CNN과 MCD의 점수가 임계를 넘었을 때의 경고입니다. 그러나 경보는 두 가지 모두를 넘어야 합니다. 즉, CNN과 MCD 모두의 점수가 임계를 넘는 경우에만 경보가 발생합니다. 반면에, 주의는 CNN과 MCD 중 하나라도 임계를 넘으면 발생합니다. 따라서 경보는 더 강력한 경고로, 주의는 경보보다는 덜 강한 경고라고 볼 수 있습니다.
참고한 근거: [F1][F2][F3][F4][F16]

Q. 임계값은 어떻게 정해?   [숫자 위반 0, 단정 0]  내 판독: 정확
A. 임계값은 학습용 정상 점수의 상위 1%로 설정됩니다.
참고한 근거: [F1][F2][F3][F4][F16]

Q. 오경보는 얼마나 자주 나?   [숫자 위반 0, 단정 0]  내 판독: 부분 — 수치는 맞으나 경보 단계 수치를 빼고 '자주 나타납니다'로 표현
A. 오경보는 CNN 단독(주의 단계) 세그먼트 오경보율이 2.8%로 자주 나타납니다.
참고한 근거: [F3][F4][F7][F9][F16]

Q. 고부하에서 오경보가 왜 많아?   [숫자 위반 0, 단정 0]  내 판독: 오류 — 수치는 맞으나 원인 서술이 뒤섞임
A. 고부하에서 오경보가 많은 이유는 CNN 단독에서 고부하 오경보율이 3.2% / 2.2%로 높아서, 이는 경보 단계(CNN AND MCD)를 지우기 위해 고부하 오경보를 제거하기 위한 조치입니다.
참고한 근거: [F3][F4][F7][F9][F16]

Q. 약한 이상도 잡아?   [숫자 위반 0, 단정 0]  내 판독: 오류 — 첫 문장이 '잡을 수 있습니다'로 근거(거의 못 잡음)와 반대
A. 약한 이상도 잡을 수 있습니다. [FACTS]에서 합성 강도 1 이하의 약한 이상은 경보도 주의도 거의 잡지 못한다고 밝혀졌습니다. 하지만 실제 이상 이벤트가 한 건이라 결과는 이 이벤트의

**발견 [데이터 + 내 판독]** 9개 중 정확 1, 부분 2, **오류 6**이다. 그런데 **코드 검사(숫자 대조·단정 표현)는 하나도 잡지 못했다.** 모델이 근거 안의 숫자를 다른 항목에 잘못 붙이거나(CNN 단독에 AND 수치), 근거와 반대로 결론을 쓰거나, 있는 근거를 "없다"고 하는 오류는 숫자 존재 여부로는 보이지 않기 때문이다.

**시사점.** 소형 모델에게 문장 생성을 맡기면 **그럴듯하지만 틀린 답**이 나오고, 지금의 안전장치로는 걸러지지 않는다. 그래서 설계를 바꿨다: 모델은 문장을 쓰지 않고 **근거 번호만 고르며**, 답은 근거 문장 그대로 출력한다(§4).

## 4. 설계 변경: 모델은 근거 번호만 고른다 (select / hybrid 모드)

표현을 일부러 키워드에서 벗어나게 쓴 질문 17개(범위 안 14, 범위 밖 3)로 **근거를 제대로 고르는지** 비교한다. 적중 = 기대 근거 번호 중 하나라도 고름, 범위 밖은 아무것도 안 고르면 적중. 정밀도 = 고른 번호 중 기대 번호의 비율.

In [5]:
RT_PATH = RES / "routing_eval.csv"
if RERUN_HF:
    hf = C.HFBackend(mode="hybrid")
    def hybrid(q):
        intent, ids = C.route(q, False)
        return ids if intent is not None else hf.select_ids(q, facts, "v2", True)
    variants = {"키워드 규칙": lambda q: C.route(q, False)[1] or [],
                "LLM v1 (번호 목록만)": lambda q: hf.select_ids(q, facts, "v1", False),
                "LLM v2 (지시·예시 추가)": lambda q: hf.select_ids(q, facts, "v2", False),
                "LLM v2 + 도메인 단어 게이트": lambda q: hf.select_ids(q, facts, "v2", True),
                "hybrid (규칙 → LLM)": hybrid}
    det = pd.concat([C.evaluate_routing(fn).assign(방식=n) for n, fn in variants.items()], ignore_index=True)
    det.to_csv(RT_PATH, index=False)
else:
    det = pd.read_csv(RT_PATH).fillna("")
summ = det.groupby("방식", sort=False).apply(lambda d: pd.Series({
    "범위 안 적중": d[d["범위"] == "안"]["적중"].mean(), "범위 밖 거절": d[d["범위"] == "밖"]["적중"].mean(), "정밀도": d["정밀도"].mean()})).round(3)
summ

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

,범위 안 적중,범위 밖 거절,정밀도
방식,,,
키워드 규칙,0.143,1.000,0.211
LLM v1 (번호 목록만),0.429,1.000,0.529
LLM v2 (지시·예시 추가),0.643,0.333,0.559
LLM v2 + 도메인 단어 게이트,0.643,1.000,0.676
hybrid (규칙 → LLM),0.643,1.000,0.593


In [6]:
det.pivot_table(index=["범위", "질문", "기대"], columns="방식", values="선택", aggfunc="first").fillna("-")[list(summ.index)]

방식                                              키워드 규칙 LLM v1 (번호 목록만) LLM v2 (지시·예시 추가) LLM v2 + 도메인 단어 게이트 hybrid (규칙 → LLM)
범위 질문                            기대                                                                                           
밖  오늘 환율이 얼마야?                                                                                                                
   점심 메뉴 추천해줘                                                                        F10                                      
   파이썬에서 리스트 정렬하는 방법 알려줘                                                             F10                                      
안  기준값은 뭘 보고 잡은 거야?              F2        F1,F2,F3,F4              F2                F2                  F2       F1,F2,F3,F4
   모터가 문제인지 펌프가 문제인지 알 수 있어?     F15       F15,F12,F13                               F17                 F17       F15,F12,F13
   미미한 이상 징후도 검출돼?               F6                                                  F13                 F13               F13
   부하가 큰 때에 알람이 더 잘못 울리는 이유가 뭐야? F7                                 F7                F7                  F7                F7
   설명이 가리키는 센서가 맞는지 확인했어?        F12                               F16               F12                 F12               F12
   센서 3개 중에 전류가 꼭 필요해?           F17                                                 F17                 F17               F17
   신호 3개 중 어디서 문제가 제일 자주 보였어?    F13       F3,F4,F7,F9             F13               F13                 F13       F3,F4,F7,F9
   알림이 시간당 몇 번 정도 오는 거야?         F3,F4,F9                           F3                F3                  F3                F3
   알림이 울리면 현장에서 뭘 확인하면 돼?        F14                               F14               F14                 F14               F14
   어떤 상황에서 알림이 울리는지 알려줘          F1                                                  F15                 F15               F15
   예측 모델이랑 통계 모델을 같이 쓰는 이유가 뭐야?  F1,F4,F5                                            F15                 F15               F15
   이 분석을 그대로 믿어도 되는 거야?          F16                                                 F16                 F16               F16
   이상이 생기기 전에 얼마나 먼저 알려줘?        F8                                 F8               F15                 F15               F15
   정상인데 잘못 울리는 비율이 어느 정도야?       F3,F4                                       F1,F3,F4,F5         F1,F3,F4,F5       F1,F3,F4,F5

**발견 [데이터]**
- 표현을 바꾼 질문에서 키워드 규칙은 범위 안 14개 중 **2개**(14%)만 맞힌다. LLM은 v1 43%, v2 64%, 도메인 단어 게이트를 더하면 범위 밖 질문을 전부 걸러내 **범위 안 64%·범위 밖 거절 100%**다.
- v2는 "조금이라도 관련 있으면 고르라"는 지시로 범위 안 적중을 올렸지만 범위 밖 질문(점심 메뉴 등)에도 근거를 골라 거절률이 떨어졌다. 도메인 단어 게이트가 이를 막았다. 이 게이트와 v2 프롬프트는 이 질문 세트를 보고 고친 것이라 **적중률은 낙관적**이다.
- 범위 안 14개 중 5개는 여전히 놓치거나 엉뚱한 근거를 고른다(예: "알림이 울리는 상황" → 센서 위치 근거, "통계 모델을 같이 쓰는 이유" → 센서 위치 근거). 1.5B 모델의 한계다. 틀려도 **근거 문장 그대로**가 나오므로 거짓 문장이 생기지는 않는다.
- hybrid는 키워드로 확실히 아는 질문은 규칙이 답하고 나머지만 LLM에 맡긴다. 아래 14개 점검은 **전부 키워드 규칙에 걸리는 질문이라 LLM이 쓰이지 않는다**(8초, 14/14). 그래서 이 표는 LLM의 증거가 아니라 hybrid 배선이 규칙 경로를 해치지 않는다는 확인이다. LLM의 증거는 위 17개 표현 변형 질문이다.

In [7]:
EV_PATH = RES / "eval_hf_hybrid.csv"
if RERUN_HF:
    import time
    t0 = time.time()
    ev_hf = C.evaluate(C.HFBackend(mode="hybrid"), facts, cards)
    ev_hf.drop(columns=["모델 원답"]).to_csv(EV_PATH, index=False)
    print("소요", round(time.time() - t0), "초")
else:
    ev_hf = pd.read_csv(EV_PATH)
print("통과", int(ev_hf["통과"].sum()), "/", len(ev_hf), " 근거 문장으로 대체된 답", int(ev_hf["근거 문장으로 대체"].sum()))
ev_hf[["질문", "종류", "인용", "거절", "근거 문장으로 대체", "통과"]]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

소요 6 초
통과 14 / 14  근거 문장으로 대체된 답 0


,질문,종류,인용,거절,근거 문장으로 대체,통과
0,경보랑 주의는 뭐가 달라?,answer,"F1,F2,F3,F4",False,False,True
1,임계값은 어떻게 정해?,answer,"F1,F2,F3,F4",False,False,True
2,오경보는 얼마나 자주 나?,answer,"F3,F4,F7,F9",False,False,True
3,고부하에서 오경보가 왜 많아?,answer,"F3,F4,F7,F9",False,False,True
4,약한 이상도 잡아?,answer,"F6,F16,F17",False,False,True
5,조기경보는 몇 분 전에 울려?,answer,"F8,F11,F16",False,False,True
6,미탐은 어떤 경우야?,answer,"F6,F16,F17",False,False,True
7,어떻게 점검해야 해?,answer,"F14,F15",False,False,True
8,이 결과를 얼마나 믿을 수 있어?,answer,"F12,F15,F16",False,False,True
9,CNN 단독이랑 MCD 단독 비교해줘,answer,"F3,F4,F5",False,False,True


## 5. 안전장치 점검: 모델이 규칙을 어기는 답을 냈을 때

모델 대신 **미리 정한 답을 돌려주는 대역**을 끼워서, 숫자 검증·단정 표현 검사가 잡고 근거 문장으로 대체하는지 본다. (§3의 오류 유형은 이 검사로 잡히지 않는다는 점이 한계다.)

In [8]:
class _Stub:
    name = "stub"
    def __init__(self, reply): self.reply = reply
    def answer(self, question, history, facts, card): return self.reply

CASES = [("정상 답(근거 안)", "경보 단계 오경보는 시간당 1.0 / 0.4건입니다 [F4]"),
         ("없는 숫자를 만든 답", "오경보는 시간당 99.9건입니다 [F3]"),
         ("원인을 단정한 답", "이 경보는 모터 베어링 고장입니다 [F13]"),
         ("시점을 예측한 답", "약 3분 후 경보 수준에 도달합니다 [F8]")]
rows = []
for name, reply in CASES:
    r = C.ChatSession(_Stub(reply), facts, cards).ask("오경보 얼마나 나?")
    rows.append({"사례": name, "위반 감지": r["violation"], "근거 문장으로 대체": r["fallback"], "최종 답 첫 줄": r["answer"].splitlines()[0]})
gt = pd.DataFrame(rows)
gt.to_csv(RES / "guardrail_test.csv", index=False)
gt

,사례,위반 감지,근거 문장으로 대체,최종 답 첫 줄
0,정상 답(근거 안),True,True,(모델 답이 근거 검사를 통과하지 못해 근거 문장으로 대체)
1,없는 숫자를 만든 답,True,True,(모델 답이 근거 검사를 통과하지 못해 근거 문장으로 대체)
2,원인을 단정한 답,True,True,(모델 답이 근거 검사를 통과하지 못해 근거 문장으로 대체)
3,시점을 예측한 답,True,True,(모델 답이 근거 검사를 통과하지 못해 근거 문장으로 대체)


**발견 [데이터]** 근거 안의 답은 그대로 통과하고, 없는 숫자(99.9)·원인 단정·시점 예측은 모두 감지되어 근거 문장으로 대체된다. 단정 표현 검사는 정규식 휴리스틱이라 다른 표현은 놓칠 수 있다.

## 6. 대화 예시: 경보 카드 선택 (hybrid 백엔드)

In [9]:
rows = []
backend = C.HFBackend(mode="hybrid") if HF_OK else C.OfflineBackend()
for label, idx in [("실제 이상 창", int(cards.index[cards.label == 1][0])), ("AND 오경보 창", int(cards.index[cards.label == 0][0]))]:
    s = C.ChatSession(backend, facts, cards)
    s.select_card(idx)
    for q in ["이 경보 왜 울렸어?", "어떻게 점검해야 해?", "고장 원인이 모터 베어링이야?"]:
        r = s.ask(q)
        rows.append({"카드": label, "질문": q, "답": r["answer"], "인용": ",".join(r["cited"])})
dlg = pd.DataFrame(rows)
dlg.to_csv(RES / "dialogue.csv", index=False)
for _, r in dlg.iterrows():
    print("[" + r["카드"] + "] Q. " + r["질문"])
    print("A. " + r["답"] + "\n")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[실제 이상 창] Q. 이 경보 왜 울렸어?
A. 선택한 경보의 근거 카드입니다.
[경보] 상부 진동 쪽 이상 의심. CNN 점수가 임계의 33.5배, MCD 점수가 임계의 2.1배. 예측 오차 비중: 상부 진동 69% · 하부 진동 14% · 전류 16%. 정상에서 가장 벗어난 통계: 상부 진동 RMS +22.1σ, 상부 진동 피크-피크 +20.7σ, 상부 진동 피크 +17.8σ. 운전 상태 추정: 저부하. 점검 권고(가정): 상단 체결부·커플링·정렬.
예측 오차가 몰린 신호는 '상부 진동'이지만 이것이 원인 부위라는 뜻은 아닙니다 [F13][F15]. 점검 권고는 가정입니다 [F14].

[실제 이상 창] Q. 어떻게 점검해야 해?
A. 근거 요약입니다.
- 점검 권고(가정): 상부 진동 우세 → 상단 체결부·커플링·정렬, 하부 진동 우세 → 하단 베이스·마운트, 전류 우세 → 전원·부하 변동·토출 압력, 진동과 전류 동시 → 구동부 전반 종합 점검. 가이드북과 일반 설비 상식에서 온 가정이며 이 데이터로 검증하지 않았다. [F14]
- 센서 부착 부위(모터 몸체인지 펌프 몸체인지)를 데이터로 확정할 수 없어 원인이 모터인지 펌프인지, 어떤 부품인지는 구분할 수 없다. 탐지 대상은 '유압펌프 모터-펌프 구동부 이상'으로 쓴다. [F15]

[실제 이상 창] Q. 고장 원인이 모터 베어링이야?
A. 원인(모터/펌프/부품)은 이 데이터로 구분할 수 없습니다. 근거로 말할 수 있는 것은 아래입니다.
- 센서 부착 부위(모터 몸체인지 펌프 몸체인지)를 데이터로 확정할 수 없어 원인이 모터인지 펌프인지, 어떤 부품인지는 구분할 수 없다. 탐지 대상은 '유압펌프 모터-펌프 구동부 이상'으로 쓴다. [F15]
- 근거 카드 정합성: 합성 이상에서 CNN 예측 오차 비중 1위 채널이 주입한 채널과 일치한 비율은 강도 2 이상의 유형·강도별 칸에서 94~100%다(합성 검증이며 실제 원인 진단 성능이 아니다). [F12]
- 실제 이상 창 63개의 오차 비중 1위 채널은 

**발견 [데이터]** 같은 질문에 카드가 바뀌면 답의 첫 부분(카드 문장)만 바뀐다. 오경보 카드는 CNN 배수 1.0 · MCD 배수 1.7로 임계 바로 위라는 점이 문장에 드러난다. 원인 질문에는 "구분할 수 없다"와 근거가 따라온다.

## 7. 해석

발견 → 시사점 → 활용은 `reports/34_alarm_chatbot_JIW.md`에 정리했다.